# 02. 기술적 지표 생성
`ohlcv.csv` → SMA, RSI, MACD, Bollinger Bands, 로그수익률, target → `features.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q scikit-learn

In [ ]:
# ── 설정 (본인 Drive 경로에 맞게 수정) ───────────────────────────────────────
DATA_DIR         = "/content/drive/MyDrive/term_project/data"
TARGET_THRESHOLD = 0.015  # 0.01 → 0.015: ±1.5% 이내 횡보 제거 (레이블 노이즈 감소)

import os
os.makedirs(DATA_DIR, exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ── 피처 생성 함수 ────────────────────────────────────────────────────────────
def add_log_return(df):
    df["log_return"] = np.log(df["close"] / df["close"].shift(1))
    return df

def add_sma(df, windows=[5, 20, 60]):
    for w in windows:
        df[f"sma_{w}"] = df["close"].rolling(window=w).mean()
    return df

def add_rsi(df, period=14):
    delta = df["close"].diff()
    gain  = delta.where(delta > 0, 0.0).rolling(window=period).mean()
    loss  = (-delta.where(delta < 0, 0.0)).rolling(window=period).mean()
    df["rsi"] = 100 - (100 / (1 + gain / loss))
    return df

def add_macd(df, fast=12, slow=26, signal=9):
    ema_fast = df["close"].ewm(span=fast).mean()
    ema_slow = df["close"].ewm(span=slow).mean()
    df["macd"]        = ema_fast - ema_slow
    df["macd_signal"] = df["macd"].ewm(span=signal).mean()
    df["macd_hist"]   = df["macd"] - df["macd_signal"]
    return df

def add_bollinger(df, window=20, num_std=2):
    sma = df["close"].rolling(window=window).mean()
    std = df["close"].rolling(window=window).std()
    df["bb_upper"] = sma + num_std * std
    df["bb_lower"] = sma - num_std * std
    df["bb_width"] = df["bb_upper"] - df["bb_lower"]
    return df

def add_momentum(df):
    df["return_5d"]  = df["close"] / df["close"].shift(5)  - 1
    df["return_20d"] = df["close"] / df["close"].shift(20) - 1
    return df

def add_volume_ratio(df, window=20):
    df["volume_ratio_20"] = df["volume"] / df["volume"].rolling(window=window).mean()
    return df

def add_52w_position(df):
    df["high_52w_ratio"] = df["close"] / df["close"].rolling(window=252).max()
    return df

def add_volatility(df, window=20):
    df["volatility_20"] = df["log_return"].rolling(window=window).std()
    return df

def add_adx(df, period=14):
    high, low, close = df["high"], df["low"], df["close"]
    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low  - close.shift(1)).abs(),
    ], axis=1).max(axis=1)
    up   = high - high.shift(1)
    down = low.shift(1) - low
    plus_dm  = np.where((up > down) & (up > 0),   up,   0.0)
    minus_dm = np.where((down > up) & (down > 0), down, 0.0)
    tr_roll  = tr.rolling(period).mean()
    plus_di  = 100 * pd.Series(plus_dm,  index=df.index).rolling(period).mean() / (tr_roll + 1e-9)
    minus_di = 100 * pd.Series(minus_dm, index=df.index).rolling(period).mean() / (tr_roll + 1e-9)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)
    df["adx"] = dx.rolling(period).mean()
    return df

def add_obv(df):
    direction = np.sign(df["close"].diff()).fillna(0)
    df["obv"] = (df["volume"] * direction).cumsum()
    return df

def add_target(df):
    """
    종목별 분위수 기반 target 생성 (절대 임계값 대신 종목 자체 수익률 분포 사용)
    - 3일 후 수익률 상위 30% → Up (1)
    - 3일 후 수익률 하위 30% → Down (0)
    - 중간 40% → NaN (횡보 제외)
    """
    df["future_return"] = df["close"].shift(-3) / df["close"] - 1
    q30 = df["future_return"].quantile(0.30)
    q70 = df["future_return"].quantile(0.70)
    df["target"] = np.nan
    df.loc[df["future_return"] >= q70, "target"] = 1
    df.loc[df["future_return"] <= q30, "target"] = 0
    return df

def process_stock(df):
    df = df.sort_values("date").reset_index(drop=True)
    df = add_log_return(df)
    df = add_sma(df)
    df = add_rsi(df)
    df = add_macd(df)
    df = add_bollinger(df)
    df = add_momentum(df)
    df = add_volume_ratio(df)
    df = add_52w_position(df)
    df = add_volatility(df)
    df = add_adx(df)
    df = add_obv(df)
    df = add_target(df)
    return df.dropna().reset_index(drop=True)


In [ ]:
# ── 데이터 로드 ───────────────────────────────────────────────────────────────
df = pd.read_csv(f"{DATA_DIR}/ohlcv.csv", parse_dates=["date"])
print(f"ohlcv.csv: {len(df)}행, 종목 수: {df['code'].nunique()}")

In [ ]:
# ── KOSPI proxy: 날짜별 전종목 평균 수익률 계산 ───────────────────────────────
# 개별 종목 데이터만 있으므로 전종목 동일가중 평균 수익률을 시장 수익률로 사용
df["_log_ret"] = np.log(
    df.groupby("code")["close"].transform(lambda x: x / x.shift(1))
)
market_return = df.groupby("date")["_log_ret"].mean()  # Series: date → 시장 평균 수익률

# ── 종목별 피처 생성 ──────────────────────────────────────────────────────────
results = []
for code, group in df.groupby("code"):
    processed = process_stock(group.copy())

    # 시장 대비 초과 수익률 (relative_return = 종목 - 시장)
    processed = processed.join(market_return.rename("_mkt"), on="date")
    processed["relative_return"] = processed["log_return"] - processed["_mkt"]
    processed = processed.drop(columns=["_mkt"])

    results.append(processed)

result = pd.concat(results, ignore_index=True)
print(f"\n전체: {len(result)}행, 종목 수: {result['code'].nunique()}")

In [ ]:
# ── 저장 ──────────────────────────────────────────────────────────────────────
output_path = f"{DATA_DIR}/features.csv"
result.to_csv(output_path, index=False, encoding="utf-8-sig")
print(f"저장 완료: {output_path} ({len(result)}행)")
result.head()